# Generative AI: chunking and retrieval experiments

Exploration only. The service depends on `nw/policy`; this notebook imports it.

## What a chunk carries

In [1]:
from pathlib import Path
from nw.policy.chunking import chunk_corpus
chunks = chunk_corpus(Path("../data/policies"))
c = next(x for x in chunks if x.doc_id == "sla-2025" and "Response" in x.section)
print(c.id, c.effective, c.audience, "current" if c.current else "superseded")
print(c.text[:600])

sla-2025#3cf55c9eebdc 2025-03-01 customer current
Service Level Agreement / Support Response Time Targets

Support requests are triaged by priority level. Response time targets, measured from the time a ticket is submitted through the support portal to the time a qualified engineer provides an initial substantive response, are as follows:

| Priority | Description | Enterprise | Pro | Starter |
|----------|--------------------------------------------|------------|---------|---------|
| P0 | Critical: service unavailable or data loss | 15 minutes | 1 hour | 4 hours |
| P1 | High: major function impaired | 1 hour | 4 hours | 8 hours |
| P2 | Me


## Chunk size against chunk count and token spread

In [2]:
import statistics
for mt in (150, 350, 700, 1200):
    cs = chunk_corpus(Path("../data/policies"), max_tokens=mt)
    toks = [x.tokens for x in cs]
    print(f"max_tokens={mt:5d}: {len(cs):4d} chunks, median {statistics.median(toks):.0f} tokens, max {max(toks)}")

max_tokens=  150:  329 chunks, median 126 tokens, max 301
max_tokens=  350:  212 chunks, median 178 tokens, max 354
max_tokens=  700:  212 chunks, median 178 tokens, max 354
max_tokens= 1200:  212 chunks, median 178 tokens, max 354


## Dense against lexical against hybrid on one exact-term query

In [3]:
from nw.policy.retrieval import PolicyIndex, real_embeddings
idx = PolicyIndex.load(Path("../artifacts/policy"), real_embeddings(), Path("../artifacts/policy/chunks.jsonl"))
q = "What does sla-2023 say about service credits?"
for name, hits in [("dense", idx.dense(q, 5)), ("bm25", idx.lexical(q, 5)), ("hybrid", idx.retrieve(q, k=5, rerank=False, current_only=False))]:
    print(name, [(h.chunk.doc_id, h.chunk.section[:22]) for h in hits[:3]])

/Users/shadabhussain/Downloads/Courses/AI Engineering/solution/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10591.42it/s]

dense [('sla-2023', 'Service Level Agreemen'), ('sla-2025', 'Service Level Agreemen'), ('sla-2023', 'Service Credits')]
bm25 [('refund-approval-internal', 'Purpose and Scope'), ('outage-communication', 'Service Credits'), ('mobile-app-support', 'Mobile App Support')]
hybrid [('outage-communication', 'Service Credits'), ('sla-2023', 'Service Level Agreemen'), ('sla-2025', 'Service Level Agreemen')]


## Reading the harness output

In [4]:
import json
r = json.load(open("../artifacts/policy/eval.json"))
agg = r["aggregate"]
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in agg.items()})
refused_wrong = [x for x in r["results"] if not x["refusal_correct"]]
print(len(refused_wrong), "cases with the wrong refusal decision")
for x in refused_wrong[:5]:
    print(" ", x["id"], "refused" if x["refused"] else "answered", "|", x["answer"][:120])

{'n': 66, 'recall_at_k': 0.955, 'recall_at_1': 0.879, 'recall_at_3': 0.917, 'mrr': 0.923, 'faithfulness': None, 'faithfulness_n': 0, 'key_fact_correctness': None, 'key_fact_n': 0, 'citation_recall': None, 'stale_facts': 0, 'citation_validity': 1.0, 'refusal_correct': 0.667, 'must_refuse_caught': 0, 'latency_p50_ms': 47.042, 'latency_p95_ms': 53.853, 'cost_usd_total': 0.0, 'cost_usd_per_case': 0.0, 'intervals': {'recall_full': {'k': 63, 'n': 66, 'rate': 0.9545454545454546, 'lo': 0.8746668588675778, 'hi': 0.9844217256655664}, 'recall_at_1_hit': {'k': 56, 'n': 66, 'rate': 0.8484848484848485, 'lo': 0.7430709513021234, 'hi': 0.9155636301732872}, 'refusal_correct': {'k': 44, 'n': 66, 'rate': 0.6666666666666666, 'lo': 0.5465634388771762, 'hi': 0.7684357087849766}, 'must_refuse_caught': {'k': 0, 'n': 22, 'rate': 0.0, 'lo': 0.0, 'hi': 0.14865487460862087}, 'key_facts_complete': {'k': 0, 'n': 0, 'rate': None, 'lo': 0.0, 'hi': 1.0}}, 'by_slice': {'answerable': {'n': 44, 'recall_at_k': 0.931818181

## Questions to bring back

- Which chunk size gave the best recall, and what did it do to p95 latency?
- For the wrong refusal decisions: was it retrieval, the model, or the citation check?